In [2]:
import matplotlib.pyplot as plt
from pathlib import Path
from monai.transforms import (
    Compose, 
    LoadImaged, 
    EnsureChannelFirstd,
    Spacingd, 
    Orientationd, 
    ScaleIntensityRanged
)

# 1. Define paths (using the same file as before)
data_dir = Path("../data/raw")
img_path = data_dir / "imagesTr" / "img0001.nii.gz"
lbl_path = data_dir / "labelsTr" / "label0001.nii.gz"

# 2. Define the MONAI preprocessing pipeline
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"], 
        pixdim=(1.5, 1.5, 2.0), 
        mode=("bilinear", "nearest")
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-175.0, a_max=250.0,
        b_min=0.0, b_max=1.0,
        clip=True
    )
])

# 3. Create a dictionary representing one sample and apply transforms
data_sample = {"image": img_path, "label": lbl_path}
transformed_data = train_transforms(data_sample)

# 4. Check results
print(f"Transformed Image Shape: {transformed_data['image'].shape}")
print(f"Transformed Label Shape: {transformed_data['label'].shape}")
print(f"New Image Min/Max: {transformed_data['image'].min():.2f} / {transformed_data['image'].max():.2f}")

Transformed Image Shape: torch.Size([1, 229, 229, 220])
Transformed Label Shape: torch.Size([1, 229, 229, 220])
New Image Min/Max: 0.00 / 1.00


In [3]:
from monai.data import CacheDataset, DataLoader

# 1. Automatically list all images and labels in our directories
images = sorted(list((data_dir / "imagesTr").glob("*.nii.gz")))
labels = sorted(list((data_dir / "labelsTr").glob("*.nii.gz")))

# Create a list of dictionaries mapping each image to its label
data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]

# For this notebook demonstration, we will only use the first 4 samples 
# to avoid long caching times.
train_files = data_dicts[:4]
print(f"Number of samples for demonstration: {len(train_files)}")

# 2. Initialize the CacheDataset
# cache_rate=1.0 stores all 4 preprocessed samples in RAM.
# num_workers=0 is crucial on Windows in Jupyter to avoid multiprocessing freezes.
train_ds = CacheDataset(
    data=train_files, 
    transform=train_transforms, 
    cache_rate=1.0, 
    num_workers=0
)

# 3. Initialize the PyTorch DataLoader
# batch_size=1 is standard for 3D medical imaging to avoid Out-Of-Memory (OOM) GPU errors.
train_loader = DataLoader(
    train_ds, 
    batch_size=1, 
    shuffle=True, 
    num_workers=0
)

# 4. Fetch one batch to verify the pipeline
train_batch = next(iter(train_loader))

print("\n--- DataLoader Batch Verification ---")
# The shape will have an extra dimension at the beginning for the batch size (B, C, X, Y, Z)
print(f"Batch Image Shape: {train_batch['image'].shape}") 
print(f"Batch Label Shape: {train_batch['label'].shape}")

Number of samples for demonstration: 4


Loading dataset: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:34<00:00,  8.66s/it]


--- DataLoader Batch Verification ---
Batch Image Shape: torch.Size([1, 1, 247, 247, 208])
Batch Label Shape: torch.Size([1, 1, 247, 247, 208])


In [1]:
import sys

sys.path.append("..")

In [3]:
from src.data.transforms import get_train_transforms

pipeline = get_train_transforms()
print("Pipeline de transformations chargé avec succès depuis src/ !")

Pipeline de transformations chargé avec succès depuis src/ !
